In [1]:
import os
import rioxarray
import numpy as np
import xarray as xr
import geopandas as gpd
from rasterio import features

In [2]:
# Define paths
DATA_DIR = os.path.join("..", "data", "ethiopia")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

In [3]:
# Load woredas
woredas_fp = os.path.join(DATA_DIR, "woredas.json")
woredas_gdf = gpd.read_file(woredas_fp)
woredas_gdf = woredas_gdf.to_crs(4326)
woredas_gdf.head()

,GID_3,GID_0,COUNTRY,GID_1,NAME_1,NL_NAME_1,GID_2,NAME_2,NL_NAME_2,NAME_3,VARNAME_3,NL_NAME_3,TYPE_3,ENGTYPE_3,CC_3,HASC_3,geometry
0,ETH.1.1.1_1,ETH,Ethiopia,ETH.1_1,AddisAbeba,NA,ETH.1.1_1,AddisAbeba,NA,AddisKetema,NA,NA,Woreda,District,140108,NA,"MULTIPOLYGON (((38.735 9.0475, 38.7351 9.0428,..."
1,ETH.1.1.2_1,ETH,Ethiopia,ETH.1_1,AddisAbeba,NA,ETH.1.1_1,AddisAbeba,NA,Akaki-Kalit,NA,NA,Woreda,District,140101,NA,"MULTIPOLYGON (((38.7728 8.9517, 38.7711 8.9452..."
2,ETH.1.1.3_1,ETH,Ethiopia,ETH.1_1,AddisAbeba,NA,ETH.1.1_1,AddisAbeba,NA,Arada,NA,NA,Woreda,District,140109,NA,"MULTIPOLYGON (((38.7438 9.0306, 38.7351 9.0428..."
3,ETH.1.1.4_1,ETH,Ethiopia,ETH.1_1,AddisAbeba,NA,ETH.1.1_1,AddisAbeba,NA,Bole,NA,NA,Woreda,District,140104,NA,"MULTIPOLYGON (((38.8556 8.9383, 38.8494 8.9383..."
4,ETH.1.1.5_1,ETH,Ethiopia,ETH.1_1,AddisAbeba,NA,ETH.1.1_1,AddisAbeba,NA,Gulele,NA,NA,Woreda,District,140110,NA,"MULTIPOLYGON (((38.735 9.0475, 38.7283 9.0459,..."


In [4]:
# Create a numeric woreda_id that maps to NAME_3
woredas_gdf['woreda_id'] = woredas_gdf.index + 1  # +1 so 0 can be "no woreda"
woredas_gdf.head()

,GID_3,GID_0,COUNTRY,GID_1,NAME_1,NL_NAME_1,GID_2,NAME_2,NL_NAME_2,NAME_3,VARNAME_3,NL_NAME_3,TYPE_3,ENGTYPE_3,CC_3,HASC_3,geometry,woreda_id
0,ETH.1.1.1_1,ETH,Ethiopia,ETH.1_1,AddisAbeba,NA,ETH.1.1_1,AddisAbeba,NA,AddisKetema,NA,NA,Woreda,District,140108,NA,"MULTIPOLYGON (((38.735 9.0475, 38.7351 9.0428,...",1
1,ETH.1.1.2_1,ETH,Ethiopia,ETH.1_1,AddisAbeba,NA,ETH.1.1_1,AddisAbeba,NA,Akaki-Kalit,NA,NA,Woreda,District,140101,NA,"MULTIPOLYGON (((38.7728 8.9517, 38.7711 8.9452...",2
2,ETH.1.1.3_1,ETH,Ethiopia,ETH.1_1,AddisAbeba,NA,ETH.1.1_1,AddisAbeba,NA,Arada,NA,NA,Woreda,District,140109,NA,"MULTIPOLYGON (((38.7438 9.0306, 38.7351 9.0428...",3
3,ETH.1.1.4_1,ETH,Ethiopia,ETH.1_1,AddisAbeba,NA,ETH.1.1_1,AddisAbeba,NA,Bole,NA,NA,Woreda,District,140104,NA,"MULTIPOLYGON (((38.8556 8.9383, 38.8494 8.9383...",4
4,ETH.1.1.5_1,ETH,Ethiopia,ETH.1_1,AddisAbeba,NA,ETH.1.1_1,AddisAbeba,NA,Gulele,NA,NA,Woreda,District,140110,NA,"MULTIPOLYGON (((38.735 9.0475, 38.7283 9.0459,...",5


In [5]:
# load GSFAD crop cover data
crop_cover_fp = os.path.join(DATA_DIR, "crop_cover", "crop_cover.vrt")
crop_cover_classes = rioxarray.open_rasterio(crop_cover_fp, chunks="auto", crs=4326)
crop_cover_classes = crop_cover_classes.squeeze('band', drop=True)
crop_cover = (crop_cover_classes == 2) # value of 2 indicates crop cover
crop_cover

<xarray.DataArray (y: 74482, x: 74222)> Size: 6GB
dask.array<eq, shape=(74482, 74222), dtype=bool, chunksize=(11520, 11520), chunktype=numpy.ndarray>
Coordinates:
  * x            (x) float64 594kB 30.0 30.0 30.0 30.0 ... 50.0 50.0 50.0 50.0
  * y            (y) float64 596kB 20.07 20.07 20.07 ... -0.0006737 -0.0009432
    spatial_ref  int64 8B 0

In [6]:
# Get the spatial extent from woredas bounds
bounds = woredas_gdf.total_bounds  # (minx, miny, maxx, maxy)
minx, miny, maxx, maxy = bounds
bounds

array([33.0015,  3.3988, 47.9582, 14.8455])

In [7]:
# Slice crop_cover to bounds
crop_cover = crop_cover.rio.clip_box(
    minx=minx,
    miny=miny,
    maxx=maxx,
    maxy=maxy
)
crop_cover

<xarray.DataArray (y: 42476, x: 55500)> Size: 2GB
dask.array<getitem, shape=(42476, 55500), dtype=bool, chunksize=(11520, 11520), chunktype=numpy.ndarray>
Coordinates:
  * x            (x) float64 444kB 33.0 33.0 33.0 33.0 ... 47.96 47.96 47.96
  * y            (y) float64 340kB 14.85 14.85 14.84 14.84 ... 3.399 3.399 3.399
    spatial_ref  int64 8B 0

In [8]:
# Create shapes iterator (geometry, value pairs)
shapes = ((geom, value) for geom, value in zip(woredas_gdf.geometry, woredas_gdf.woreda_id))

# Rasterize using the crop_cover template for transform and shape
woredas_rasterized = features.rasterize(
    shapes=shapes,
    out_shape=crop_cover.shape,
    transform=crop_cover.rio.transform(),
    fill=0,  # Background value for non-woreda areas
    dtype='int16'
)

# Convert to DataArray with same coords as crop_cover
woredas_da = xr.DataArray(
    woredas_rasterized,
    coords=crop_cover.coords,
    dims=crop_cover.dims,
    name='woreda_id'
)

woredas_da

<xarray.DataArray 'woreda_id' (y: 42476, x: 55500)> Size: 5GB
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(42476, 55500), dtype=int16)
Coordinates:
  * x            (x) float64 444kB 33.0 33.0 33.0 33.0 ... 47.96 47.96 47.96
  * y            (y) float64 340kB 14.85 14.85 14.84 14.84 ... 3.399 3.399 3.399
    spatial_ref  int64 8B 0

In [9]:
# Combine both DataArrays into a single Dataset
woreda_crop_cover = xr.Dataset({
    'woreda_id': woredas_da,
    'crop_cover': crop_cover
})

woreda_crop_cover

<xarray.Dataset> Size: 7GB
Dimensions:      (x: 55500, y: 42476)
Coordinates:
  * x            (x) float64 444kB 33.0 33.0 33.0 33.0 ... 47.96 47.96 47.96
  * y            (y) float64 340kB 14.85 14.85 14.84 14.84 ... 3.399 3.399 3.399
    spatial_ref  int64 8B 0
Data variables:
    woreda_id    (y, x) int16 5GB 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0
    crop_cover   (y, x) bool 2GB dask.array<chunksize=(3649, 378), meta=np.ndarray>